<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook assembles the analysis table used for Firth logistic regression / PheWAS of locityper-genotyped structural variants. It combines locityper genotypes with All of Us survey answers, ZIP3 socioeconomic measures, short-read genomic QC metrics, and OMOP condition, observation, measurement, drug, and procedure records, then annotates concepts with OHDSI vocabulary.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import pandas as pd
from google.cloud import bigquery

from datetime import date

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Inputs from the notebook's data location: GWAS results (gwas1.csv) and Locityper genotyping results.
# The Locityper table's 'source' column is used below to pick the PheWAS samples.
!gsutil cp {AOU_LR_DATA}/gwas1.csv gwas1.csv

In [ ]:
!gsutil cp {AOU_LR_DATA}/locityper_genotyping_v2.csv.gz "locityper_genotyping_v2.csv.gz"

In [ ]:
gwas_df = pd.read_csv("gwas1.csv")

In [ ]:
gwas_df

In [ ]:
lt_df = pd.read_csv("locityper_genotyping_v2.csv.gz")

In [ ]:
lt_df

In [ ]:
# Dataset Builder queries for all short-read participants with EHR data: survey answers, ZIP3
# socioeconomic data (median income, poverty, deprivation index), and demographics.
import pandas
import os

# This query represents dataset "All participants, basic info, socioeconomic status" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_87959485_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1586134, 40192389)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_ehr_data = 1 ) )
        )"""

dataset_87959485_survey_df = pandas.read_gbq(
    dataset_87959485_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_87959485_survey_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "All participants, basic info, socioeconomic status" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v7
dataset_87959485_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) ) 
        AND observation_source_concept_id = 1585250 
        AND observation.value_as_string NOT LIKE 'Res%'"""

dataset_87959485_zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_87959485_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_87959485_zip_code_socioeconomic_df['person_id'] = dataset_87959485_zip_code_socioeconomic_df['person_id'].astype(str)

dataset_87959485_zip_code_socioeconomic_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "All participants, basic info, socioeconomic status" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_87959485_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

dataset_87959485_person_df = pandas.read_gbq(
    dataset_87959485_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_87959485_person_df.head(5)

In [ ]:
# Short-read QC metrics table, used for each participant's sex at birth and sequencing site.
if not os.path.exists("genomic_metrics.tsv"):
    !gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv .
    
sr_metrics = pd.read_table('genomic_metrics.tsv')

In [ ]:
sr_metrics

In [ ]:
# Samples in the SV PheWAS set (rows whose source is Sample_sv.pheWAS.txt.gz).
sample_ids = set(lt_df[lt_df['source'] == 'Sample_sv.pheWAS.txt.gz']['sample'])
sample_ids = { str(x) for x in sample_ids }

In [ ]:
# Pull every concept recorded for those samples from five OMOP domains (conditions, observations,
# measurements, drug exposures, procedures).
client = bigquery.Client()
CDR_version = os.getenv("WORKSPACE_CDR")

sample_ids_str = ",".join(sample_ids)

query = f"""
WITH combined_data AS (
    SELECT person_id, condition_concept_id AS concept_id, 'condition' AS source
    FROM `{CDR_version}.condition_occurrence`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, observation_concept_id AS concept_id, 'observation' AS source
    FROM `{CDR_version}.observation`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, measurement_concept_id AS concept_id, 'measurement' AS source
    FROM `{CDR_version}.measurement`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, drug_concept_id AS concept_id, 'drug_exposure' AS source
    FROM `{CDR_version}.drug_exposure`
    WHERE person_id IN ({sample_ids_str})

    UNION ALL

    SELECT person_id, procedure_concept_id AS concept_id, 'procedure' AS source
    FROM `{CDR_version}.procedure_occurrence`
    WHERE person_id IN ({sample_ids_str})
)
SELECT person_id, concept_id, source FROM combined_data
"""

df_samples = client.query(query).to_dataframe().drop_duplicates()
#output_file = "AoU_LR_samples_all_domains.csv"
#df_samples.to_csv(output_file, index=False)

In [ ]:
df_samples[df_samples['source'] == 'condition']

In [ ]:
# Add sequencing site and sex at birth to each row.
# Merge to add sex_at_birth from sr_metrics to df_samples
df_samples = df_samples.merge(
    sr_metrics[["research_id", "site_id", "sex_at_birth"]],
    left_on="person_id",
    right_on="research_id",
    how="left"
)

# Optionally drop the extra research_id column after merge
df_samples.drop(columns=["research_id"], inplace=True)

In [ ]:
df_samples

In [ ]:
# OHDSI vocabulary tables, stored in the workspace bucket.
!gsutil -u $GOOGLE_PROJECT cp $WORKSPACE_BUCKET/resources/vocabulary.zip .

In [ ]:
!unzip -o vocabulary.zip

In [ ]:
# Keep condition concepts and attach concept names and classes.
df_conditions = df_samples[df_samples["source"] == "condition"][["person_id", "concept_id", "site_id", "sex_at_birth"]]

vocabulary_file = "CONCEPT.csv"  # from OHDSI
df_vocab = pd.read_csv(vocabulary_file, sep="\t",low_memory=False)

df_vocab = df_vocab[["concept_id", "concept_name", "concept_class_id", "standard_concept"]]
df_merged = pd.merge(df_conditions, df_vocab, on="concept_id", how="left")

In [ ]:
df_merged['person_id'] = df_merged['person_id'].astype(str)

In [ ]:
df_merged

In [ ]:
# Restrict to the PheWAS samples.
df_subset = df_merged[df_merged["person_id"].isin(sample_ids)]

In [ ]:
df_subset

In [ ]:
dataset_87959485_person_df['person_id'] = dataset_87959485_person_df['person_id'].astype(str)

In [ ]:
# Age in whole years as of today.
# Ensure the column is datetime
dataset_87959485_person_df["date_of_birth"] = pd.to_datetime(dataset_87959485_person_df["date_of_birth"], errors="coerce")

# Today's date
today = pd.Timestamp(date.today())

# Compute age in years
dataset_87959485_person_df["age"] = dataset_87959485_person_df["date_of_birth"].apply(
    lambda dob: today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))
)

In [ ]:
# Build the covariate table for the regression: each condition row with age, and the ZIP3 median
# income, poverty fraction, and deprivation index.
df_with_age = pd.merge(
    df_subset,
    dataset_87959485_person_df[["person_id", "age"]],
    on="person_id",
    how="left"   # keeps all rows from df_subset
)

df_with_di = pd.merge(
    df_with_age,
    dataset_87959485_zip_code_socioeconomic_df[["person_id", "median_income", "poverty", "deprivation_index"]],
    on="person_id",
    how="left"
)

In [ ]:
# Drop concept bookkeeping columns, leaving one row per participant-condition with covariates.
df_final = df_with_di.drop(columns=["concept_id", "concept_class_id", "standard_concept"])

In [ ]:
df_final